# CHSA Medical Triage Agent - Google Colab T4

This notebook is the Colab/T4 version of the training workflow. It avoids creating a `.venv`, uses Colab's built-in PyTorch/CUDA stack, downloads the private Hugging Face dataset, audits it, then runs full SFT and DPO adapter training with commented smoke commands kept for quick checks.

It can be run directly in Colab or from VSCode connected to a Colab Jupyter runtime. Use a T4 GPU runtime; uncomment a smoke command first when validating a fresh runtime.

## Runtime checks

The repository targets Python 3.12+. Stop early if the runtime is older.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

python_version = tuple(int(part) for part in sys.version.split()[0].split(".")[:2])
print("python=", sys.version)
if python_version < (3, 12):
    raise RuntimeError(
        "This project requires Python 3.12+. Use a Colab runtime with Python 3.12 or run on Kaggle."
    )

python= 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
import torch

print("cuda_available=", torch.cuda.is_available())
print("device=", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("torch=", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab before training.")

cuda_available= True
device= Tesla T4
torch= 2.11.0+cu128


## Clone or refresh the repository

This is idempotent. If the repo already exists, it pulls the latest `main`.

In [3]:
import importlib.util

REPO_URL = "https://github.com/Nhkp/medical-triage-agent.git"
if importlib.util.find_spec("google.colab"):
    DEFAULT_REPO_DIR = Path("/content/medical-triage-agent")
else:
    DEFAULT_REPO_DIR = Path.cwd() / ".colab_runtime" / "medical-triage-agent"

REPO_DIR = DEFAULT_REPO_DIR

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif Path.cwd().name == "medical-triage-agent" and (Path.cwd() / ".git").exists():
    REPO_DIR = Path.cwd()
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout; remove it or choose another path"
        )
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(Path.cwd())

/content/medical-triage-agent


## Install only missing training dependencies

Do not reinstall Torch. Colab already provides the CUDA-enabled build, and replacing it can waste time or break the runtime.

In [4]:
!python -m pip install -q -U \
  datasets peft trl transformers accelerate bitsandbytes pyyaml trackio wrapt huggingface_hub

## Load Hugging Face token

Preferred: put `HF_TOKEN` in a local `.env` file at the repository root. Fallbacks: Colab Secrets, then secure prompt. The token needs read access to the private dataset.

In [5]:
from getpass import getpass


def load_dotenv(path: Path) -> dict[str, str]:
    values = {}
    if not path.exists():
        return values
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


dotenv = load_dotenv(Path(".env"))
token = dotenv.get("HF_TOKEN")

if not token:
    try:
        from google.colab import userdata
        from google.colab.userdata import SecretNotFoundError, TimeoutException

        token = userdata.get("HF_TOKEN")
    except (ImportError, KeyError, SecretNotFoundError, TimeoutException):
        token = None

if not token:
    token = getpass("HF_TOKEN: ")

os.environ["HF_TOKEN"] = token
os.environ["HF_DATASET_REPO"] = dotenv.get("HF_DATASET_REPO", "Lokhidor/medical-triage-dataset")
os.environ.setdefault(
    "HF_SFT_MODEL_REPO", dotenv.get("HF_SFT_MODEL_REPO", "Lokhidor/medical-triage-qwen3-sft-lora")
)
os.environ.setdefault(
    "HF_DPO_MODEL_REPO", dotenv.get("HF_DPO_MODEL_REPO", "Lokhidor/medical-triage-qwen3-dpo-lora")
)
print("HF token loaded:", bool(os.environ.get("HF_TOKEN")))

HF token loaded: True


## Download the private dataset

The training configs expect local files in `data/processed/training`.

In [6]:
from huggingface_hub import snapshot_download

DATA_DIR = Path("data/processed/training")
DATA_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=os.environ["HF_DATASET_REPO"],
    repo_type="dataset",
    local_dir=DATA_DIR,
    token=os.environ["HF_TOKEN"],
    allow_patterns=["sft_*.jsonl", "dpo_*.jsonl", "manifest.json", "README.md"],
)
print("downloaded files:")
for path in sorted(DATA_DIR.glob("*")):
    print("-", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

downloaded files:
- data/processed/training/.cache
- data/processed/training/README.md
- data/processed/training/dpo_clinical_eval.jsonl
- data/processed/training/dpo_test.jsonl
- data/processed/training/dpo_train.jsonl
- data/processed/training/dpo_validation.jsonl
- data/processed/training/manifest.json
- data/processed/training/sft_clinical_eval.jsonl
- data/processed/training/sft_test.jsonl
- data/processed/training/sft_train.jsonl
- data/processed/training/sft_validation.jsonl


## Audit downloaded data

This validates schema, provenance, duplicates, split isolation, and obvious PII findings before training.

In [7]:
!PYTHONPATH=src python -m medical_triage_agent audit-training-data data/processed/training
!PYTHONPATH=src python -m medical_triage_agent summarize-training-data data/processed/training

{
  "generated_at": "2026-08-12T16:04:09Z",
  "passed": true,
  "errors": [
    "missing clinical_review_queue.jsonl"
  ],
  "accepted_counts": {
    "sft": 5000,
    "dpo": 1000
  },
  "rejected_counts": {
    "mediqa": 763,
    "frenchmedmcqa": 1,
    "medquad": 53,
    "ultramedical_preference": 2
  },
  "pii_findings": 0,
  "duplicate_findings": 0,
  "missing_provenance_findings": 0,
  "language_counts": {
    "en": 3406,
    "fr": 2594
  },
  "source_counts": {
    "frenchmedmcqa": 594,
    "mediqa": 2000,
    "medquad": 2406,
    "ultramedical_preference": 1000
  },
  "split_counts": {
    "sft": {
      "train": 4001,
      "validation": 490,
      "test": 357,
      "clinical_eval": 152
    },
    "dpo": {
      "train": 791,
      "validation": 111,
      "test": 65,
      "clinical_eval": 33
    }
  },
  "transforms": [
    "load_hf:ANR-MALADES/MediQAl:oeq:test",
    "load_hf:TsinghuaC3I/UltraMedical-Preference:train",
    "load_hf:keivalya/MedQuad-MedicalQnADataset:train",
 

## SFT full run on T4

Run the full SFT pass after dataset audit succeeds. The adapter is pushed to `HF_SFT_MODEL_REPO`.

In [8]:
!python scripts/train_sft.py \
  --config configs/sft.yaml \
  --push-to-hub \
  --hub-model-id "$HF_SFT_MODEL_REPO"

Loading weights: 100% 310/310 [00:04<00:00, 74.59it/s]
Adding EOS to train dataset: 100% 4001/4001 [00:00<00:00, 34388.42 examples/s]
Tokenizing train dataset: 100% 4001/4001 [00:04<00:00, 846.52 examples/s] 
Building labels for train dataset: 100% 4001/4001 [00:00<00:00, 5280.64 examples/s]
Truncating train dataset: 100% 4001/4001 [00:00<00:00, 7609.92 examples/s]
Dropping fully masked examples from train dataset: 100% 4001/4001 [00:00<00:00, 31980.12 examples/s]
Adding EOS to eval dataset: 100% 490/490 [00:00<00:00, 20012.94 examples/s]
Tokenizing eval dataset: 100% 490/490 [00:00<00:00, 823.72 examples/s] 
Building labels for eval dataset: 100% 490/490 [00:00<00:00, 4562.08 examples/s]
Truncating eval dataset: 100% 490/490 [00:00<00:00, 8431.73 examples/s]
Dropping fully masked examples from eval dataset: 100% 490/490 [00:00<00:00, 33309.71 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config an

## SFT smoke run

Keep this commented smoke command for quick runtime checks before a full run.

In [9]:
# !python scripts/train_sft.py \
#   --config configs/sft.yaml \
#   --max-steps 5 \
#   --max-train-samples 32

## DPO full run after SFT

Run DPO only after `outputs/sft` exists. The aligned adapter is pushed to `HF_DPO_MODEL_REPO`.

In [10]:
!python scripts/train_dpo.py \
  --config configs/dpo.yaml \
  --push-to-hub \
  --hub-model-id "$HF_DPO_MODEL_REPO"

Loading weights: 100% 310/310 [00:05<00:00, 59.41it/s]
Adding EOS to train dataset: 100% 791/791 [00:00<00:00, 10900.49 examples/s]
Tokenizing train dataset:   0% 0/791 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejecte

## DPO smoke run

Keep this commented smoke command for quick DPO startup checks.

In [11]:
# !python scripts/train_dpo.py \
#   --config configs/dpo.yaml \
#   --max-steps 5 \
#   --max-train-samples 32

## Optional: deterministic evaluation

Use after the SFT adapter exists.

In [12]:
!python scripts/evaluate.py \
  --config configs/sft.yaml \
  --model sft \
  --adapter-path outputs/sft \
  --output outputs/evaluations/sft.json

Loading weights: 100% 310/310 [00:01<00:00, 218.30it/s]
wrote outputs/evaluations/sft.json


In [ ]:
# from google.colab import files
# import shutil

# shutil.make_archive("outputs", "zip", "outputs")
# files.download("outputs.zip")